# Sample Document Corpus Walkthrough — `src/storage/`

End-to-end showcase of the **synthetic medical-bill + salvage-claim corpus** added for
analysis and fine-tuning without proprietary American Family Insurance data.

This notebook covers the new pipelines:

1. Rich skeleton schemas (medical + salvage)
2. Realistic sample generation (HCFA / UB-04 / LOG / sales / towing)
3. SQLite store seeding
4. Claim bundles (multi-document files)
5. Export → DICIE inference
6. Import existing fixtures / eval gold

| Section | What you will see |
|---------|-------------------|
| 0 | Setup |
| 1 | Schemas & taxonomies |
| 2 | Generate synthetic corpus |
| 3 | Seed SQLite store |
| 4 | Inspect claims, docs, fields |
| 5 | Claim bundles |
| 6 | Export for DICIE / training |
| 7 | Run DICIE on corpus export |
| 8 | Import fixtures into the store |
| 9 | Provenance |

Canonical docs: [`docs/sample_document_corpus.md`](../docs/sample_document_corpus.md),
[`docs/data_provenance.md`](../docs/data_provenance.md).

> All documents are **fictional**. Carrier branding (`American Family` / `AmFam`) is used
> only to simulate intake surfaces — no proprietary claim files are loaded.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import sqlite3
import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / "pyproject.toml").exists() else CWD.parent
assert (REPO_ROOT / "pyproject.toml").exists(), f"Could not find repo root from {CWD}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.docie import DociePipeline
from src.docie.applications import list_applications, load_application
from src.docie.eval import evaluate_application
from src.docie.pipeline import run_file
from src.storage import DocumentStore
from src.storage.sample_generator import generate_claim_bundle, generate_corpus
from src.storage.schema import DDL, SCHEMA_VERSION
from src.storage.training import (
    fit_tfidf_random_forest,
    prepare_both_applications,
)
from src.storage.types import ClaimRecord, DocumentRecord, FieldRecord
from src.utils.config import Config
from src.utils.io import load_jsonl, read_json, write_json

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
cfg = Config.load()

DEMO = REPO_ROOT / "data" / "notebook_demo" / "sample_corpus"
DEMO.mkdir(parents=True, exist_ok=True)
DB_PATH = DEMO / "documents.db"
EXPORTS = DEMO / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)
PREPARED = DEMO / "prepared"
MODELS = DEMO / "models"
MODELS.mkdir(parents=True, exist_ok=True)

SEED = 42
print(f"repo:     {REPO_ROOT}")
print(f"demo db:  {DB_PATH}")
print(f"exports:  {EXPORTS}")
print(f"schema v: {SCHEMA_VERSION}")
print(f"apps:     {list_applications()}")

## 1. Schemas & taxonomies

In [ ]:
med_schema = read_json(REPO_ROOT / "data" / "schemas" / "medical_bill_skeleton.schema.json")
sal_schema = read_json(REPO_ROOT / "data" / "schemas" / "salvage_document_skeleton.schema.json")
print("Medical schema title:", med_schema["title"])
print("  required:", med_schema["required"])
print("Salvage schema title:", sal_schema["title"])
print("  required:", sal_schema["required"])

for app in ("medical_bills", "salvage_claims"):
    profile = load_application(app)
    print(f"\n[{app}] labels={profile.labels}")
    print(f"  extraction_fields={profile.extraction_fields}")

## 2. Generate synthetic corpus

In [ ]:
corpus = generate_corpus(
    seed=SEED,
    medical_per_type=6,
    salvage_per_type=6,
    bundles_per_app=2,
    include_canonical_fixtures=True,
)
print(f"claims={len(corpus.claims)}  documents={len(corpus.documents)}")

type_counts = Counter((d.application, d.document_type) for d in corpus.documents)
pd.DataFrame(
    [{"application": a, "document_type": t, "count": n} for (a, t), n in sorted(type_counts.items())]
)

### Peek at a Letter of Guarantee and an HCFA

In [ ]:
log_doc = next(d for d in corpus.documents if d.document_type == "log")
hcfa_doc = next(d for d in corpus.documents if d.document_type == "hcfa")
display(Markdown(f"**{log_doc.document_id}** (`{log_doc.document_type}`)"))
print(log_doc.text[:800])
print("---")
display(Markdown(f"**{hcfa_doc.document_id}** (`{hcfa_doc.document_type}`)"))
print(hcfa_doc.text[:800])
print("\nLOG ground truth:", log_doc.ground_truth_fields())
print("HCFA ground truth:", hcfa_doc.ground_truth_fields())

## 3. Seed SQLite store

In [ ]:
if DB_PATH.exists():
    DB_PATH.unlink()
store = DocumentStore(DB_PATH)
n = store.bulk_upsert(corpus.documents, claims=corpus.claims)
store.add_provenance(
    stage="notebook_seed",
    source="sample_document_corpus_walkthrough",
    detail={"documents": n, "seed": SEED},
)
summary = store.summary()
display(summary)
pd.DataFrame(summary["by_application_type"])

## 4. Inspect claims, documents, and fields

In [ ]:
docs = store.list_documents(limit=8)
pd.DataFrame(
    [
        {
            "document_id": d.document_id,
            "application": d.application,
            "document_type": d.document_type,
            "claim_id": d.claim_id,
            "split": d.split,
            "source_kind": d.source_kind,
            "n_fields": len(d.fields),
        }
        for d in docs
    ]
)

In [ ]:
canonical = store.get_document("sal-log-001")
assert canonical is not None
claim = store.get_claim(canonical.claim_id) if canonical.claim_id else None
print("carrier:", claim.carrier_name if claim else None)
print("fields:", canonical.ground_truth_fields())
print(canonical.text)

## 5. Claim bundles (multi-document salvage / medical files)

In [ ]:
import random

rng = random.Random(7)
claim, bundle_docs = generate_claim_bundle(rng, application="salvage_claims", bundle_index=99)
store.upsert_claim(claim)
for d in bundle_docs:
    store.upsert_document(d)

bundled = store.list_documents(claim_id=claim.claim_id)
print(f"claim {claim.claim_id} has {len(bundled)} documents:")
for d in bundled:
    print(f"  - {d.document_id:28s} {d.document_type:8s}  title={d.title}")

## 6. Export for DICIE / training

In [ ]:
salvage_docie = EXPORTS / "salvage_docie.jsonl"
medical_docie = EXPORTS / "medical_docie.jsonl"
clf_all = EXPORTS / "classification_all.jsonl"
ext_all = EXPORTS / "extraction_all.jsonl"

n_sal = store.export_jsonl(salvage_docie, format="docie", application="salvage_claims")
n_med = store.export_jsonl(medical_docie, format="docie", application="medical_bills")
n_clf = store.export_jsonl(clf_all, format="classification")
n_ext = store.export_jsonl(ext_all, format="extraction")
print({"salvage_docie": n_sal, "medical_docie": n_med, "classification": n_clf, "extraction": n_ext})

sample = load_jsonl(salvage_docie)[0]
meta = {k: sample[k] for k in sample if k != "text"}
print(json.dumps(meta, indent=2)[:800])
print("text preview:\n", sample["text"][:400])

## 7. Run DICIE on corpus export

In [ ]:
out_pred = DEMO / "docie_salvage_predictions.jsonl"
run_file(
    salvage_docie,
    out_pred,
    application="salvage_claims",
    cfg=cfg,
    run_ocr=False,
    limit=20,
)
summary_path = out_pred.with_suffix(".summary.json")
print("summary →", summary_path)
pred_summary = read_json(summary_path)
display(pred_summary)

preds = load_jsonl(out_pred)
pd.DataFrame(
    [
        {
            "record_id": p["record_id"],
            "document_type": p.get("document_type"),
            "needs_human_review": p.get("needs_human_review"),
            "fields": p.get("fields"),
        }
        for p in preds[:12]
    ]
)

### Single-document DICIE on a generated LOG

In [ ]:
pipe = DociePipeline(application="salvage_claims", cfg=cfg, run_ocr=False)
demo_log = store.get_document("sal-log-001")
prediction = pipe.process(record_id=demo_log.document_id, text=demo_log.text)
display(prediction.response_payload())

## 8. Import existing fixtures / eval gold into the store

In [ ]:
fixtures = REPO_ROOT / "tests" / "fixtures" / "sample_docie_documents.jsonl"
eval_set = REPO_ROOT / "data" / "eval" / "docie_eval_set.jsonl"
n_fix = store.import_docie_jsonl(fixtures, source_kind="test_fixture")
n_eval = store.import_docie_jsonl(eval_set, source_kind="docie_eval_gold") if eval_set.exists() else 0
print(f"imported fixtures={n_fix} eval_gold={n_eval}")
store.summary()

## 9. Provenance

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        "SELECT event_id, stage, source, detail_json, created_at "
        "FROM provenance_events ORDER BY event_id"
    ).fetchall()
pd.DataFrame([dict(r) for r in rows])

## Next notebooks

- [`sample_corpus_sql_integrations.ipynb`](sample_corpus_sql_integrations.ipynb) — every SQL integration surface
- [`sample_corpus_train_test_pipeline.ipynb`](sample_corpus_train_test_pipeline.ipynb) — full train → test pipeline from the store